In [3]:
import os
import requests
import google.generativeai as genai
import pdfplumber
from groq import Groq
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [9]:
system_prompt = """
### ROLE
You are a senior academic researcher and scientific paper analyst.

### CONTEXT
The input you receive is the extracted full text of a downloaded scientific journal article (from PDF). 
The text may contain formatting noise, broken sentences, or incomplete structure due to PDF extraction.

### TASK
Analyze and summarize the scientific journal accurately and concisely.

### INSTRUCTIONS

1. First, identify the paper structure if available:
   - Title
   - Abstract
   - Introduction / Background
   - Methodology
   - Results
   - Discussion
   - Conclusion

2. Extract and summarize:
   - Research problem
   - Objective
   - Method / model used
   - Dataset or sample
   - Experimental setup
   - Evaluation metrics
   - Key findings (retain numerical results if available)
   - Main contributions
   - Limitations
   - Future work (if available)

3. If sections are missing or unclear, state:
   "Not clearly specified in the provided text."

4. Ignore:
   - References section
   - Copyright text
   - Author affiliations
   - Formatting artifacts

5. Do NOT fabricate information.
6. Maintain academic tone.
7. Be concise but comprehensive.

### OUTPUT FORMAT

Title:
Field:
Problem Statement:
Objective:
Methodology:
Dataset:
Evaluation Metrics:
Key Results:
Contributions:
Limitations:
Conclusion:
"""

In [4]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [7]:
def extract_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text

article_text = extract_pdf(r"C:\Users\istiq\Downloads\1015.pdf")
print(article_text[:2000])

Volume 5, No. 10 October, 2024
p-ISSN 2721-3854 | e-ISSN 2721-2769
Formulation and Stability of Exfoliating Liquid Soap Nanoemulsion
from Lerak Extract (Sapindus rarak DC) and Coconut Dregs Waste
Istiqamah Harnama1*
Universitas Diponegoro, Indonesia
Email: istiqamahharnamabirrul@gmail.com
ABSTRACT
The use of natural-based skincare products is increasing along with public awareness of the importance of
choosing safe and environmentally friendly ingredients. This study aims to develop a nanoemulsion-based
exfoliating liquid soap from lerak (Sapindus rarak DC) extract and coconut pulp. Considering the
environmental challenges due to the use of plastic-based products, this study adopts an eco-friendly
approach by using natural raw materials. The Ultrasonic-Assisted Extraction (UAE) method was applied to
extract active compounds from lerak fruit, followed by the preparation of nanoemulsions using a high-
pressure homogenizer. Results showed that the resulting nanoemulsion had good stability

In [10]:
def summarize_article(article_text):

    article_text = article_text[:12000]   

    full_user_prompt = f"""
Please read and summarize the following scientific article.

ARTICLE TEXT:
-----------------------
{article_text}
-----------------------

Focus on:
- Core research problem
- Method or approach
- Key findings (include numbers if available)
- Why this research matters

Output format:
Hook:
Core Insight:
Practical Takeaway:
"""

    response = model.generate_content(
    [
        system_prompt,
        full_user_prompt
    ]
)

    return response

In [11]:
article_text = extract_pdf(r"C:\Users\istiq\Downloads\1015.pdf")

response = summarize_article(article_text)

display(Markdown(response.text))

Title: Formulation and Stability of Exfoliating Liquid Soap Nanoemulsion from Lerak Extract (Sapindus rarak DC) and Coconut Dregs Waste
Field: Cosmetic Science, Green Chemistry, Pharmaceutical Technology

Problem Statement: The increasing public awareness of safe and environmentally friendly products highlights the need for natural-based skincare alternatives. Traditional synthetic surfactants and microplastics used in cosmetic products pose environmental challenges (waste generation, pollution) and health risks (cytotoxicity, skin irritation).

Objective: To develop a nanoemulsion-based exfoliating liquid soap utilizing lerak (Sapindus rarak DC) extract as a natural surfactant and coconut pulp (Cocos Nucifera L.) waste as a natural exfoliator, and to assess its formulation stability and safety for potential use as an eco-friendly cosmetic product.

Methodology:
1.  **Extraction:** Ultrasonic-Assisted Extraction (UAE) was applied to extract active compounds (saponins) from lerak fruit, chosen for its efficiency, reduced solvent use, and preservation of heat-sensitive compounds.
2.  **Nanoemulsion Preparation:** Nanoemulsions were prepared using a high-pressure homogenizer, a high-energy technique favored for its simplicity, high stability, small particle size dispersion, and compositional flexibility.
3.  **Evaluation:** The stability of the nanoemulsion was assessed based on droplet size, viscosity, and homogeneity. Product shelf life and physical form were tested using a centrifugator. Organoleptic tests were conducted to evaluate sensory properties and skin safety (irritant effects).

Dataset: The primary raw materials used were lerak (Sapindus rarak DC) fruit extract and coconut pulp (Cocos Nucifera L.) waste.

Evaluation Metrics: Droplet size, viscosity, homogeneity, physical form stability (via centrifugator), organoleptic properties, and skin irritancy.

Key Results:
*   The resulting nanoemulsion demonstrated good stability.
*   It exhibited a small droplet size and viscosity suitable for cosmetic product applications.
*   Organoleptic tests confirmed that the final product was safe and non-irritating to the skin.

Contributions:
*   Successful development of an environmentally friendly exfoliating liquid soap utilizing indigenous Indonesian natural resources (lerak and coconut pulp).
*   Offers a sustainable alternative to synthetic surfactant-based products and microplastic exfoliators.
*   Contributes to the development of sustainable cosmetic products and raises awareness regarding the use of natural ingredients in the beauty industry.

Limitations: Not clearly specified in the provided text. Detailed experimental parameters (e.g., concentrations, specific homogenizer settings, precise stability test protocols, comparative data against commercial products) were not elaborated upon.

Conclusion: This study successfully formulated a stable, safe, and non-irritating nanoemulsion-based exfoliating liquid soap using lerak extract and coconut pulp waste. The product shows significant potential as a sustainable and eco-friendly alternative in the cosmetic industry, aligning with green entrepreneurship principles and utilizing local biodiversity.

JOURNAL COMPARISON

In [12]:
system_prompt = """
### ROLE
You are a senior scientific comparative analyst.

### CORE MISSION
You MUST analyze ALL provided articles (A, B, and C if available).
Failure to analyze all articles is considered an incomplete response.

### CRITICAL INSTRUCTION
Before writing the comparison:
1. Identify the main topic of Paper A.
2. Identify the main topic of Paper B.
3. Confirm internally that BOTH are different studies.
4. Only then proceed with comparison.

If one paper is ignored, explicitly state:
"Paper X content not sufficiently detected."

### ANALYSIS RULES
- Do NOT fabricate information.
- Preserve numerical values exactly as written.
- If something is missing, write:
  "Not specified in provided text."
- Focus only on core scientific contributions.
- Ignore references, DOIs, and formatting noise.
- Maintain formal academic tone.
- DO NOT summarize only one article.

### OUTPUT FORMAT (MANDATORY)

Step 1 – Individual Identification:
Paper A Main Focus:
Paper B Main Focus:
Paper C Main Focus (if provided):

Step 2 – Structured Summaries:
Paper A Summary:
Paper B Summary:
Paper C Summary (if provided):

Step 3 – Comparative Analysis:
- Research Objective Differences:
- Methodological Differences:
- Dataset / Sample Differences:
- Evaluation Metrics Differences:
- Key Performance Results Comparison:
- Strengths & Weaknesses Contrast:
- Research Gaps Contrast:

Step 4 – Strategic Insight:
(High-level analytical interpretation)
"""

In [19]:
model ="llama-3.3-70b-versatile"

In [18]:
def compare_articles(article_a, article_b, article_c=None):

    article_a = article_a[:5000]
    article_b = article_b[:5000]
    article_c = article_c[:5000] if article_c else ""

    full_user_prompt = f"""
You are given multiple scientific journal articles.

Your task is to compare them analytically.

==============================
ARTICLE A
==============================
{article_a}

==============================
ARTICLE B
==============================
{article_b}
"""

    if article_c:
        full_user_prompt += f"""
==============================
ARTICLE C
==============================
{article_c}
"""

    full_user_prompt += """
IMPORTANT:
- You must analyze BOTH Article A and Article B.
- If you only describe one article, the response is invalid.
- Perform structured comparison as defined in the system instructions.
"""

    client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY")
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": full_user_prompt}
        ]
    )

    return response

In [20]:
article_a = extract_pdf(r"C:\Users\istiq\OneDrive - Universitas Diponegoro\2024\aa PENELITIAN TERAPAN\Ide dari KKN\Semua jurnal-jurnal sapindus\1015.pdf")
article_b = extract_pdf(r"C:\Users\istiq\OneDrive - Universitas Diponegoro\2024\aa PENELITIAN TERAPAN\Ide dari KKN\Semua jurnal-jurnal sapindus\1111.pdf")

response = compare_articles(article_a, article_b)

display(Markdown(response.choices[0].message.content))

### Step 1 – Individual Identification:
- Paper A Main Focus: Development of a nanoemulsion-based exfoliating liquid soap from lerak (Sapindus rarak DC) extract and coconut pulp, focusing on an eco-friendly approach.
- Paper B Main Focus: Development and stability of intimate soap formulations using Sapindus saponaria L. extract as a natural surfactant, analyzing its stability and surfactant characteristics.
- Paper C Main Focus: Not provided.

### Step 2 – Structured Summaries:
- Paper A Summary: This study focuses on creating a sustainable cosmetic product by formulating a nanoemulsion-based exfoliating liquid soap using natural ingredients like lerak extract and coconut pulp. It aims to provide an environmentally friendly alternative to synthetic surfactant-based products, utilizing the Ultrasonic-Assisted Extraction method and high-pressure homogenization to create a stable nanoemulsion.
- Paper B Summary: This research develops intimate soap formulations using Sapindus saponaria L. extract as a natural surfactant. It evaluates the stability, surfactant characteristics, and other properties of the formulations, concluding that the incorporation of S. saponaria extract is an excellent option for reducing the use of synthetic anionic surfactants in liquid soap formulations.
- Paper C Summary: Not applicable.

### Step 3 – Comparative Analysis:
- Research Objective Differences: The primary objective of Paper A is to develop an eco-friendly exfoliating liquid soap using natural ingredients, whereas Paper B focuses on creating intimate soap formulations with a natural surfactant, specifically targeting the reduction of synthetic surfactants.
- Methodological Differences: Paper A uses the Ultrasonic-Assisted Extraction method to extract active compounds from lerak fruit and prepares nanoemulsions using a high-pressure homogenizer. In contrast, Paper B evaluates the stability and characteristics of intimate soap formulations using S. saponaria extract without specifying the extraction method.
- Dataset / Sample Differences: Paper A involves the use of lerak extract and coconut pulp, while Paper B uses S. saponaria extract. The specific datasets or sample sizes are not compared directly in the provided texts.
- Evaluation Metrics Differences: Paper A assesses the stability and safety of the nanoemulsion, including organoleptic tests. Paper B evaluates preliminary and accelerated stability parameters, rheological characteristics, surface tension, foaming power, foam stability, and emulsification potential.
- Key Performance Results Comparison: Both studies conclude that the use of natural extracts (lerak in Paper A and S. saponaria in Paper B) provides a viable and eco-friendly alternative to synthetic surfactants, with Paper A focusing on the stability of the nanoemulsion and Paper B on the surfactant characteristics of the formulations.
- Strengths & Weaknesses Contrast: A strength of Paper A is its focus on using abundant natural resources for eco-friendly products, while a potential weakness could be the scalability of the Ultrasonic-Assisted Extraction method. A strength of Paper B is the detailed evaluation of the surfactant characteristics, but a weakness might be the lack of information on the extraction process.
- Research Gaps Contrast: Both studies contribute to the development of sustainable cosmetic products but highlight different research gaps. Paper A suggests the need for more research on utilizing local, natural resources for cosmetics, while Paper B implies a gap in the use of natural surfactants in intimate soap formulations.

### Step 4 – Strategic Insight:
The development of eco-friendly cosmetic products, particularly those utilizing natural surfactants, is a growing area of research. Both Paper A and Paper B demonstrate the feasibility of using natural extracts as alternatives to synthetic surfactants, each targeting different applications (exfoliating liquid soap and intimate soap formulations, respectively). The key takeaway is the potential for natural ingredients to not only provide sustainable solutions but also to offer effective and safe alternatives in the cosmetics industry. This insight suggests a strategic direction for future research and development: focusing on the exploration of diverse natural resources for cosmetic applications, enhancing extraction and formulation methods, and evaluating the long-term sustainability and market viability of such products.